In [1]:
import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap

from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV,
    cross_val_score, LeaveOneGroupOut
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, recall_score, precision_score
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

sns.set_theme(style="whitegrid", font="DejaVu Sans", font_scale=1.1)
GREEN, RED, AMBER = "#2D6A4F", "#DC2626", "#F59E0B"

os.makedirs("artefacts/plots", exist_ok=True)

print("*" *20)
print("Teacher Attrition EWS")
print("*" *20)

********************
Teacher Attrition EWS
********************


# LOAD & VALIDATE DATA

In [2]:


DATA_PATH = "/content/bulletin_data_statistics.csv"

REQUIRED_COLS = [
    "Province", "Year",
    "Teacher_count_primary",
    "PTR_primary", # bulletin value — used only for cross-check
    "Student_enrolment_primary",
    "Primary_Schools",
    "Rural_schools",
    "Urban_schools",
]

if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
    print(f"\nLoaded  : {DATA_PATH}")
    print(f"Shape   : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
else:
    raise FileNotFoundError(
        f"Data file '{DATA_PATH}' not found.\n"
        "Place your extracted bulletin CSV in the same folder as this notebook."
    )

# Basic validation
missing_cols = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Standardise province names (strip whitespace, title case)
df_raw["Province"] = df_raw["Province"].str.strip().str.title()
df_raw["Year"]     = df_raw["Year"].astype(int)

print(f"\nProvinces : {sorted(df_raw['Province'].unique())}")
print(f"Years     : {df_raw['Year'].min()} – {df_raw['Year'].max()}")
print(f"\nRows per province:")
print(df_raw.groupby("Province")["Year"].count().to_string())
print(f"\nMissing values per column:")
print(df_raw[REQUIRED_COLS].isnull().sum().to_string())

#  Cross-check: recalculated PTR vs bulletin PTR
df_raw["ptr_check"] = (
    df_raw["Student_enrolment_primary"] / df_raw["Teacher_count_primary"]
).round(2)
df_raw["ptr_diff"]  = (df_raw["ptr_check"] - df_raw["PTR_primary"]).abs()

large_diff = df_raw[df_raw["ptr_diff"] > 10][
    ["Province","Year","PTR_primary","ptr_check","ptr_diff"]
]
if len(large_diff):
    print(f"\nWARNING — {len(large_diff)} rows where bulletin PTR differs from "
          "calculated PTR by >10 points:")
    print(large_diff.to_string(index=False))
    print("The notebook will use the CALCULATED PTR (enrolment / teachers).")
else:
    print("\nPTR cross-check passed — bulletin values consistent with calculated values.")

df_raw.drop(columns=["ptr_check","ptr_diff"], inplace=True)


Loaded  : /content/bulletin_data_statistics.csv
Shape   : 136 rows × 8 columns

Provinces : ['Central', 'Copperbelt', 'Eastern', 'Luapula', 'Lusaka', 'Muchinga', 'North-Western', 'Northern', 'Southern', 'Western']
Years     : 2009 – 4800

Rows per province:
Province
Central          14
Copperbelt       14
Eastern          14
Luapula          14
Lusaka           14
Muchinga         10
North-Western    14
Northern         14
Southern         14
Western          14

Missing values per column:
Province                     0
Year                         0
Teacher_count_primary        0
PTR_primary                  1
Student_enrolment_primary    0
Primary_Schools              0
Rural_schools                0
Urban_schools                1

WARNING — 65 rows where bulletin PTR differs from calculated PTR by >10 points:
     Province  Year  PTR_primary  ptr_check  ptr_diff
      Central  2010         49.2      38.66     10.54
      Central  2012         55.8      38.34     17.46
      Central

# Feature Engineering

In [3]:
df = df_raw.copy()
df = df.sort_values(["Province","Year"]).reset_index(drop=True)

# ── 2.1  Recalculated PTR (internally consistent) ─────────────
# Formula : student_enrolment_primary / teacher_count_primary
# Why     : Bulletin PTR values are inconsistently computed across
#           years (some use double-shifting adjustments, some don't).
#           This version is always computed the same way.
df["ptr_primary_calc"] = (
    df["Student_enrolment_primary"] / df["Teacher_count_primary"]
).round(4)

# ── 2.2  Teacher Growth Rate ───────────────────────────────────
# Formula : (teachers_t - teachers_{t-1}) / teachers_{t-1} × 100
# Why     : Captures whether the province is growing or shrinking
#           its primary teacher workforce year-on-year.
#           Negative = net loss of teachers (attrition signal).
df["teacher_growth_rate"] = (
    df.groupby("Province")["Teacher_count_primary"]
    .pct_change() * 100
).round(4)

# ── 2.3  Enrolment Growth Rate ────────────────────────────────
# Formula : (enrolment_t - enrolment_{t-1}) / enrolment_{t-1} × 100
# Why     : Rising enrolment with falling teachers = increasing
#           pressure on remaining staff, a driver of attrition.
df["enrolment_growth_rate"] = (
    df.groupby("Province")["Student_enrolment_primary"]
    .pct_change() * 100
).round(4)

# ── 2.4  Recruitment Gap ──────────────────────────────────────
# Formula : enrolment_growth_rate - teacher_growth_rate
# Why     : Captures supply-demand mismatch. A large positive value
#           means learner numbers are growing much faster than teacher
#           recruitment — provinces under this pressure are more
#           likely to lose teachers through overload and burnout.
df["recruitment_gap"] = (
    df["enrolment_growth_rate"] - df["teacher_growth_rate"]
).round(4)

# ── 2.5  Teachers per School ──────────────────────────────────
# Formula : teacher_count_primary / Primary_Schools
# Why     : Staffing density. Very low values indicate that schools
#           are severely understaffed, raising individual workload
#           and increasing risk of departure.
df["teachers_per_school"] = (
    df["Teacher_count_primary"] / df["Primary_Schools"]
).round(4)

# ── 2.6  Learners per School ──────────────────────────────────
# Formula : student_enrolment_primary / Primary_Schools
# Why     : Average school size. Large schools in under-resourced
#           provinces signal high stress on school leadership and
#           teachers, contributing to attrition.
df["learners_per_school"] = (
    df["Student_enrolment_primary"] / df["Primary_Schools"]
).round(4)

# ── 2.7  Rural School Percentage ──────────────────────────────
# Formula : Rural_schools / (Rural_schools + Urban_schools)
# Why     : Structural remoteness proxy. Provinces with a high
#           proportion of rural schools face teacher retention
#           challenges due to isolation, poor housing, and limited
#           amenities. Note: Rural_schools and Urban_schools are
#           aggregated across primary and secondary (bulletin data
#           does not disaggregate by level consistently).
df["rural_school_pct"] = (
    df["Rural_schools"] / (df["Rural_schools"] + df["Urban_schools"])
).round(4)

# ── 2.8  PTR Trend (3-year rolling average) ───────────────────
# Formula : rolling mean of ptr_primary_calc over 3 years
# Why     : A single-year PTR spike may be noise; a province where
#           PTR has been rising consistently for 3 years is under
#           sustained structural pressure. More predictive of future
#           attrition than point-in-time PTR.
#           min_periods=2 allows computation when only 2 years exist.
df["ptr_trend_3yr"] = (
    df.groupby("Province")["ptr_primary_calc"]
    .transform(lambda x: x.rolling(3, min_periods=2).mean())
).round(4)

# ── 2.9  Proxy attrition rate (target variable source) ────────
# Formula : max(0, -(teacher_count_t - teacher_count_{t-1}))
#           / teacher_count_{t-1}
# Why     : Since attrition counts were inconsistently recorded
#           across bulletins, we derive net teacher loss from
#           headcount change. Only provinces that LOST teachers
#           receive a positive attrition proxy rate; provinces that
#           grew their workforce receive 0 (they recruited enough
#           to offset departures).

teacher_prev = df.groupby("Province")["Teacher_count_primary"].shift(1)
net_loss     = teacher_prev - df["Teacher_count_primary"]
df["attrition_proxy_rate"] = (
    net_loss.clip(lower=0) / teacher_prev
).round(4)

print("\nFeature engineering complete.")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Preview key engineered columns ───────────────────────────
preview_cols = [
    "Province","Year",
    "Teacher_count_primary","ptr_primary_calc",
    "teacher_growth_rate","enrolment_growth_rate",
    "recruitment_gap","rural_school_pct","attrition_proxy_rate"
]
print("\nSample (first 14 rows):")
print(df[preview_cols].head(14).to_string(index=False))



Feature engineering complete.
Dataset shape: 136 rows × 17 columns

Sample (first 14 rows):
Province  Year  Teacher_count_primary  ptr_primary_calc  teacher_growth_rate  enrolment_growth_rate  recruitment_gap  rural_school_pct  attrition_proxy_rate
 Central  2009                   7136           48.8695                  NaN                    NaN              NaN            0.8783                   NaN
 Central  2010                   9098           38.6578              27.4944                 0.8534         -26.6410            0.8783                0.0000
 Central  2011                   8331           45.6723              -8.4304                 8.1849          16.6153            0.8902                0.0843
 Central  2012                   9301           38.3395              11.6433                -6.2813         -17.9246            0.8890                0.0000
 Central  2013                   9489           40.9968               2.0213                 9.0924           7.0711      

# Proxy Label Definition


In [4]:
RISK_PERCENTILE = 0.70   # top 30% = high risk

df["risk_label"] = (
    df["attrition_proxy_rate"] >= df.groupby("Year")["attrition_proxy_rate"]
    .transform(lambda x: x.quantile(RISK_PERCENTILE))
).astype(int)

# Edge case: if attrition_proxy_rate is 0 for ALL provinces in a year
# (everyone grew), override to flag the lowest-growth provinces instead
zero_attrition_years = (
    df.groupby("Year")["attrition_proxy_rate"]
    .max()[lambda x: x == 0].index.tolist()
)
if zero_attrition_years:
    print(f"\nNote: In years {zero_attrition_years}, all provinces grew their "
          "teacher workforce (attrition_proxy_rate = 0 for all).")
    print("Flagging provinces with lowest teacher_growth_rate as high risk.")
    for yr in zero_attrition_years:
        mask    = df["Year"] == yr
        tgr_col = df.loc[mask, "teacher_growth_rate"]
        thresh  = tgr_col.quantile(1 - RISK_PERCENTILE)
        df.loc[mask, "risk_label"] = (tgr_col <= thresh).astype(int)

print(f"\n── Label distribution ──────────────────────────────────")
counts = df["risk_label"].value_counts().rename({0:"not_at_risk", 1:"high_risk"})
print(counts)
print(f"High-risk rate: {df['risk_label'].mean()*100:.1f}%")

print("\nHigh-risk provinces per year:")
for yr, grp in df[df["risk_label"]==1].groupby("Year"):
    print(f"  {yr}: {', '.join(sorted(grp['Province']))}")



Note: In years [2010, 2012, 2021, 2022, 2023, 2024], all provinces grew their teacher workforce (attrition_proxy_rate = 0 for all).
Flagging provinces with lowest teacher_growth_rate as high risk.

── Label distribution ──────────────────────────────────
risk_label
not_at_risk    83
high_risk      53
Name: count, dtype: int64
High-risk rate: 39.0%

High-risk provinces per year:
  2010: Eastern, Northern, Western
  2011: Copperbelt, Lusaka, Western
  2012: Central, North-Western, Northern
  2013: Copperbelt, Eastern, Northern
  2014: Central, Luapula, Muchinga
  2016: Eastern, Luapula, Southern
  2019: Central, Copperbelt, Eastern, Luapula, Lusaka, Muchinga, Northern, Southern, Western
  2020: Central, Copperbelt, Eastern, Luapula, Lusaka, Muchinga, North-Western, Northern, Southern, Western
  2021: Copperbelt, Lusaka, Southern
  2022: Copperbelt, Lusaka, Southern
  2023: Central, Copperbelt, Lusaka
  2024: Luapula, Muchinga, Northern
  2025: Central, Copperbelt, Muchinga
  4800: North

# Exploratory Data Analysis


In [10]:
PROVINCES  = sorted(df["Province"].unique())
N_PROV     = len(PROVINCES)
PROV_COLORS= sns.color_palette("tab10", N_PROV)
PROV_CMAP  = dict(zip(PROVINCES, PROV_COLORS))

print("\n── EDA ─────────────────────────────────────────────────")

# ── 4.1  Teacher count over time by province ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for prov, grp in df.groupby("Province"):
    ax.plot(grp["Year"], grp["Teacher_count_primary"],
            marker="o", markersize=4, linewidth=1.8,
            label=prov, color=PROV_CMAP[prov])
ax.set_title("Primary Teacher Count by Province (2009–2025)", fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Primary Teachers")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# ── 4.2  PTR (calculated) over time by province ───────────────
fig, ax = plt.subplots(figsize=(12, 5))
for prov, grp in df.groupby("Province"):
    ax.plot(grp["Year"], grp["ptr_primary_calc"],
            marker="o", markersize=4, linewidth=1.8,
            label=prov, color=PROV_CMAP[prov])
ax.axhline(40, color="grey", linestyle="--", linewidth=1,
           label="UNESCO recommended PTR (40:1)")
ax.set_title("Calculated Pupil-Teacher Ratio — Primary (Gr 1-7) by Province",
             fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("PTR")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# ── 4.3  Attrition proxy rate by province ────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
df_nozero = df[df["attrition_proxy_rate"] > 0]
for prov, grp in df_nozero.groupby("Province"):
    ax.bar(grp["Year"] + list(PROVINCES).index(prov)*0.08,
           grp["attrition_proxy_rate"]*100,
           width=0.08, label=prov, color=PROV_CMAP[prov], alpha=0.85)
ax.set_title("Net Teacher Loss Rate (Proxy Attrition) by Province", fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Net Loss Rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# ── 4.4  Recruitment gap over time ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for prov, grp in df.groupby("Province"):
    grp_valid = grp.dropna(subset=["recruitment_gap"])
    ax.plot(grp_valid["Year"], grp_valid["recruitment_gap"],
            marker="o", markersize=4, linewidth=1.5,
            label=prov, color=PROV_CMAP[prov])
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Recruitment Gap (Enrolment Growth − Teacher Growth) by Province",
             fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Gap (percentage points)")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# ── 4.5  Rural school % vs attrition proxy rate ───────────────
fig, ax = plt.subplots(figsize=(7, 5))
colors = df["risk_label"].map({0: GREEN, 1: RED})
sc = ax.scatter(df["rural_school_pct"]*100, df["attrition_proxy_rate"]*100,
                c=colors, alpha=0.75, edgecolors="white",
                linewidth=0.4, s=60)
ax.set_title("Rural School % vs Net Teacher Loss Rate", fontsize=13)
ax.set_xlabel("Rural School % (primary + secondary)")
ax.set_ylabel("Net Loss Rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=RED, label="High Risk"),
                   Patch(color=GREEN, label="Not at Risk")])
plt.tight_layout()
plt.show()

# ── 4.6  Correlation heatmap ─────────────────────────────────
FEAT_PREVIEW = [
    "ptr_primary_calc","teacher_growth_rate","enrolment_growth_rate",
    "recruitment_gap","teachers_per_school","learners_per_school",
    "rural_school_pct","ptr_trend_3yr","attrition_proxy_rate",
]
corr = df[FEAT_PREVIEW].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn_r",
            center=0, ax=ax, linewidths=0.4, annot_kws={"size":8})
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

# ── 4.7  Descriptive stats by risk label ─────────────────────
print("\nDescriptive statistics by risk label:")
desc_cols = [
    "ptr_primary_calc","teacher_growth_rate","enrolment_growth_rate",
    "recruitment_gap","rural_school_pct","teachers_per_school"
]
print(df.groupby("risk_label")[desc_cols].mean().round(3).to_string())

# ── 4.8  Risk label heatmap (province × year) ────────────────
pivot = df.pivot(index="Province", columns="Year", values="risk_label")
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap=["#B7E4C7", "#DC2626"], linewidths=0.5,
            linecolor="white", ax=ax, cbar=False,
            annot=pivot.applymap(lambda v: "H" if v == 1 else ""),
            fmt="", annot_kws={"size": 8, "color": "white"})
ax.set_title("Risk Label by Province and Year  (Red = High Risk)", fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("")
plt.tight_layout()
plt.show()


── EDA ─────────────────────────────────────────────────

Descriptive statistics by risk label:
            ptr_primary_calc  teacher_growth_rate  enrolment_growth_rate  recruitment_gap  rural_school_pct  teachers_per_school
risk_label                                                                                                                      
0                     43.337               11.704                  5.236           -6.469             0.783              113.291
1                     50.702                9.254                  3.440           -5.814             0.779                9.500


# Train / Test split

In [7]:
FEATURE_COLS = [
    "ptr_primary_calc",       # Workload — enrolment / teachers
    "teacher_growth_rate",    # Workforce trajectory YoY
    "enrolment_growth_rate",  # Demand trajectory YoY
    "recruitment_gap",        # Supply-demand mismatch
    "teachers_per_school",    # Staffing density
    "learners_per_school",    # Average school size
    "rural_school_pct",       # Structural remoteness
    "ptr_trend_3yr",          # Sustained PTR pressure (3yr rolling)
]

# Drop rows where any feature OR label is NaN
# (Mainly the first 1-2 years per province — no lag values yet)
df_model = df.dropna(subset=FEATURE_COLS + ["risk_label"]).copy()

print(f"\nRows available for modelling : {len(df_model)}")
print(f"Dropped (missing lag features): {len(df) - len(df_model)}")
print(f"Class distribution:")
print(df_model["risk_label"].value_counts().rename({0:"not_at_risk",1:"high_risk"}).to_string())

# ── Temporal split ────────────────────────────────────────────
# Train: 2009–2021  |  Test: 2022–2025
# Rationale: keeps the most recent 3-4 years as held-out,
# simulating a real deployment scenario where the model predicts
# future years it has never seen.
# Note: 2015, 2017, 2018 missing from all provinces — this is
# noted in the limitations section.
TRAIN_END = 2021

train = df_model[df_model["Year"] <= TRAIN_END]
test  = df_model[df_model["Year"] >  TRAIN_END]

X_train = train[FEATURE_COLS]
y_train = train["risk_label"]
X_test  = test[FEATURE_COLS]
y_test  = test["risk_label"]
groups  = train["Province"]

print(f"\nTrain : years ≤ {TRAIN_END}  |  {len(X_train)} rows  |  "
      f"{y_train.sum()} high-risk")
print(f"Test  : years > {TRAIN_END}  |  {len(X_test)} rows   |  "
      f"{y_test.sum()} high-risk")


Rows available for modelling : 125
Dropped (missing lag features): 11
Class distribution:
risk_label
not_at_risk    73
high_risk      52

Train : years ≤ 2021  |  84 rows  |  39 high-risk
Test  : years > 2021  |  41 rows   |  13 high-risk


# Model Training

In [8]:

print("\n── XGBoost + RandomizedSearchCV ────────────────────────")

param_dist = {
    "n_estimators":     [50, 100, 150, 200, 300],
    "max_depth":        [2, 3, 4, 5],
    "learning_rate":    [0.01, 0.05, 0.1, 0.15, 0.2],
    "subsample":        [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "min_child_weight": [1, 2, 3, 5],
    "gamma":            [0, 0.1, 0.2, 0.5],
    "scale_pos_weight": [1, 2, 3],
    "reg_alpha":        [0, 0.1, 0.5],
    "reg_lambda":       [1, 1.5, 2],
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42,
    verbosity=0,
)

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring="roc_auc",
    cv=cv_strat,
    random_state=42,
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
)

print("Running RandomizedSearchCV (60 iterations, 5-fold stratified CV)...")
search.fit(X_train, y_train)
best_model = search.best_estimator_

print(f"Best CV AUC  : {search.best_score_:.4f}")
print(f"Best params  :\n{json.dumps(search.best_params_, indent=4)}")

# ── Leave-One-Province-Out CV ─────────────────────────────────
# Most rigorous evaluation for small-N longitudinal data.
# Each fold holds out ALL years of one province and trains on
# the remaining N-1 provinces. Tests whether the model generalises
# to a province it has never seen — the hardest and most realistic
# scenario.
print("\nLeave-One-Province-Out cross-validation...")
logo     = LeaveOneGroupOut()
lopo_res = []

for fold_i, (tr_idx, te_idx) in enumerate(logo.split(X_train, y_train, groups)):
    X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
    y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]
    held_out   = groups.iloc[te_idx].unique()[0]

    if len(y_te.unique()) < 2:
        print(f"  Fold {fold_i+1} ({held_out}) — skipped (only one class)")
        continue

    m = XGBClassifier(
        **search.best_params_,
        objective="binary:logistic",
        eval_metric="auc",
        use_label_encoder=False,
        random_state=42,
        verbosity=0,
    )
    m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_te)[:,1]
    pred  = (proba >= 0.65).astype(int)
    auc   = roc_auc_score(y_te, proba)
    f1    = f1_score(y_te, pred, zero_division=0)
    lopo_res.append({"province": held_out, "auc": auc, "f1": f1})
    print(f"  Held out: {held_out:<15}  AUC={auc:.3f}  F1={f1:.3f}")

lopo_df = pd.DataFrame(lopo_res)
print(f"\nLOPO AUC : {lopo_df['auc'].mean():.4f} ± {lopo_df['auc'].std():.4f}")
print(f"LOPO F1  : {lopo_df['f1'].mean():.4f}  ± {lopo_df['f1'].std():.4f}")



── XGBoost + RandomizedSearchCV ────────────────────────
Running RandomizedSearchCV (60 iterations, 5-fold stratified CV)...
Best CV AUC  : 0.6766
Best params  :
{
    "subsample": 1.0,
    "scale_pos_weight": 2,
    "reg_lambda": 2,
    "reg_alpha": 0,
    "n_estimators": 300,
    "min_child_weight": 1,
    "max_depth": 3,
    "learning_rate": 0.1,
    "gamma": 0.5,
    "colsample_bytree": 0.6
}

Leave-One-Province-Out cross-validation...
  Held out: Central          AUC=0.750  F1=0.000
  Held out: Copperbelt       AUC=0.938  F1=0.750
  Held out: Eastern          AUC=1.000  F1=0.909
  Held out: Luapula          AUC=0.650  F1=0.667
  Held out: Lusaka           AUC=0.925  F1=0.857
  Held out: Muchinga         AUC=0.500  F1=0.667
  Held out: North-Western    AUC=0.917  F1=0.667
  Held out: Northern         AUC=0.600  F1=0.727
  Held out: Southern         AUC=0.650  F1=0.667
  Held out: Western          AUC=0.750  F1=0.400

LOPO AUC : 0.7679 ± 0.1693
LOPO F1  : 0.6310  ± 0.2600


# Model Evaluation


In [ ]:

print("\n── Held-out test set evaluation ────────────────────────")

y_prob = best_model.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.65).astype(int)

test_auc  = roc_auc_score(y_test, y_prob) if len(y_test.unique()) > 1 else float("nan")
test_f1   = f1_score(y_test, y_pred, zero_division=0)
test_rec  = recall_score(y_test, y_pred, zero_division=0)
test_prec = precision_score(y_test, y_pred, zero_division=0)

print(f"ROC-AUC   : {test_auc:.4f}")
print(f"F1        : {test_f1:.4f}")
print(f"Recall    : {test_rec:.4f}")
print(f"Precision : {test_prec:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=["not_at_risk","high_risk"], zero_division=0))

# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Not at Risk","High Risk"]).plot(
    ax=ax, colorbar=False, cmap="Greens"
)
ax.set_title("Confusion Matrix — Test Set (2022–2025)")
plt.tight_layout()
plt.savefig("artefacts/plots/08_confusion_matrix.png", dpi=150)
plt.close()
print("\nSaved: 08_confusion_matrix.png")

# ── ROC curve ─────────────────────────────────────────────────
if len(y_test.unique()) > 1:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color=GREEN, lw=2,
            label=f"XGBoost (AUC = {test_auc:.3f})")
    ax.plot([0,1],[0,1],"k--",linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Test Set")
    ax.legend()
    plt.tight_layout()
    plt.savefig("artefacts/plots/09_roc_curve.png", dpi=150)
    plt.close()
    print("Saved: 09_roc_curve.png")

# ── Per province-year prediction detail ──────────────────────
test_results = test[["Province","Year","risk_label"]].copy()
test_results["risk_score"]       = y_prob.round(4)
test_results["predicted_label"]  = y_pred
test_results["correct"]          = (
    test_results["risk_label"] == test_results["predicted_label"]
).astype(int)
print("\nTest set predictions:")
print(test_results.sort_values(["Year","Province"]).to_string(index=False))



── Held-out test set evaluation ────────────────────────
ROC-AUC   : 0.4835
F1        : 0.3704
Recall    : 0.3846
Precision : 0.3571

Classification Report:
              precision    recall  f1-score   support

 not_at_risk       0.70      0.68      0.69        28
   high_risk       0.36      0.38      0.37        13

    accuracy                           0.59        41
   macro avg       0.53      0.53      0.53        41
weighted avg       0.59      0.59      0.59        41


Saved: 08_confusion_matrix.png
Saved: 09_roc_curve.png

Test set predictions:
     Province  Year  risk_label  risk_score  predicted_label  correct
      Central  2022           0      0.5078                0        1
   Copperbelt  2022           1      0.8298                1        1
      Eastern  2022           0      0.5962                0        1
      Luapula  2022           0      0.6684                1        0
       Lusaka  2022           1      0.8813                1        1
     Muchinga  2

# SHAP explainability


In [ ]:
print("\n── SHAP explainability ─────────────────────────────────")

explainer   = shap.TreeExplainer(best_model)
shap_vals   = explainer.shap_values(X_train)
mean_shap   = np.abs(shap_vals).mean(axis=0)

FEATURE_LABELS = {
    "ptr_primary_calc":       "PTR — Primary (Gr 1-7)",
    "teacher_growth_rate":    "Teacher Growth Rate (YoY %)",
    "enrolment_growth_rate":  "Enrolment Growth Rate (YoY %)",
    "recruitment_gap":        "Recruitment Gap (Enrolment−Teacher Growth)",
    "teachers_per_school":    "Teachers per School",
    "learners_per_school":    "Learners per School",
    "rural_school_pct":       "Rural School %",
    "ptr_trend_3yr":          "PTR Trend (3-Year Rolling Avg)",
}

shap_df = pd.DataFrame({
    "feature":       FEATURE_COLS,
    "feature_label": [FEATURE_LABELS[f] for f in FEATURE_COLS],
    "mean_shap":     mean_shap,
}).sort_values("mean_shap", ascending=False)

print("\nGlobal Feature Importance (Mean |SHAP|):")
print(shap_df[["feature_label","mean_shap"]].to_string(index=False))

# ── SHAP bar chart ────────────────────────────────────────────
sorted_shap = shap_df.sort_values("mean_shap", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(sorted_shap["feature_label"], sorted_shap["mean_shap"],
               color=GREEN, edgecolor="white")
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Global Feature Importance — XGBoost Province Attrition Model")
ax.set_xlim(0, sorted_shap["mean_shap"].max() * 1.18)
plt.tight_layout()
plt.savefig("artefacts/plots/10_shap_global_importance.png", dpi=150)
plt.close()
print("Saved: 10_shap_global_importance.png")

# ── Per-province SHAP for latest year ────────────────────────
latest_year = df_model["Year"].max()
latest_data = df_model[df_model["Year"] == latest_year].copy()
shap_latest = explainer.shap_values(latest_data[FEATURE_COLS])
latest_data["risk_score"] = best_model.predict_proba(
    latest_data[FEATURE_COLS])[:,1]

top_idx  = latest_data["risk_score"].idxmax()
top_prov = latest_data.loc[top_idx, "Province"]
top_pos  = latest_data.index.get_loc(top_idx)
sv       = shap_latest[top_pos]

wf_df = pd.DataFrame({
    "feature_label": [FEATURE_LABELS[f] for f in FEATURE_COLS],
    "shap":          sv,
}).sort_values("shap")

fig, ax = plt.subplots(figsize=(9, 5))
colors = [RED if v > 0 else GREEN for v in wf_df["shap"]]
bars   = ax.barh(wf_df["feature_label"], wf_df["shap"],
                 color=colors, edgecolor="white")
ax.bar_label(bars, fmt="%+.3f", padding=3, fontsize=9)
ax.axvline(0, color="#64748B", linewidth=0.8)
ax.set_xlabel("SHAP contribution to risk score")
ax.set_title(
    f"SHAP Waterfall — {top_prov} ({latest_year})\n"
    f"Risk score: {latest_data.loc[top_idx,'risk_score']:.3f}"
)
plt.tight_layout()
plt.savefig("artefacts/plots/11_shap_waterfall_top_province.png", dpi=150)
plt.close()
print(f"Saved: 11_shap_waterfall_top_province.png  ({top_prov})")


── SHAP explainability ─────────────────────────────────

Global Feature Importance (Mean |SHAP|):
                             feature_label  mean_shap
Recruitment Gap (Enrolment−Teacher Growth)   0.511379
               Teacher Growth Rate (YoY %)   0.479775
                    PTR — Primary (Gr 1-7)   0.302238
            PTR Trend (3-Year Rolling Avg)   0.243707
             Enrolment Growth Rate (YoY %)   0.180926
                       Teachers per School   0.100720
                       Learners per School   0.095544
                            Rural School %   0.083692
Saved: 10_shap_global_importance.png
Saved: 11_shap_waterfall_top_province.png  (North-Western)


# Province Predictions (latest year)

In [ ]:
print(f"\n── Province predictions for {latest_year} ───────────────────")

predictions = []
shap_all    = explainer.shap_values(latest_data[FEATURE_COLS])

for i, (idx, row) in enumerate(latest_data.iterrows()):
    sv   = shap_all[i]
    shap_dict = {
        FEATURE_LABELS[FEATURE_COLS[j]]: round(float(sv[j]), 4)
        for j in range(len(FEATURE_COLS))
    }
    risk_score = float(best_model.predict_proba(
        row[FEATURE_COLS].values.reshape(1,-1))[0,1])
    risk_label = "high_risk" if risk_score >= 0.65 else "not_at_risk"
    conf_pct   = round(
        risk_score * 100 if risk_label == "high_risk"
        else (1 - risk_score) * 100, 2
    )
    predictions.append({
        "province":           row["Province"],
        "year":               int(row["Year"]),
        "risk_score":         round(risk_score, 4),
        "risk_label":         risk_label,
        "confidence_pct":     conf_pct,
        "ptr_primary_calc":   round(row["ptr_primary_calc"], 2),
        "teacher_growth_rate":round(row["teacher_growth_rate"], 4),
        "recruitment_gap":    round(row["recruitment_gap"], 4),
        "rural_school_pct":   round(row["rural_school_pct"], 4),
        "attrition_proxy_rate": round(row["attrition_proxy_rate"], 4),
        "shap_json":          json.dumps(shap_dict),
    })

pred_df = pd.DataFrame(predictions).sort_values("risk_score", ascending=False)

print(f"\nProvince Risk Predictions — {latest_year}:")
print(pred_df[[
    "province","risk_label","risk_score","confidence_pct",
    "ptr_primary_calc","teacher_growth_rate","recruitment_gap"
]].to_string(index=False))


── Province predictions for 4800 ───────────────────

Province Risk Predictions — 4800:
     province risk_label  risk_score  confidence_pct  ptr_primary_calc  teacher_growth_rate  recruitment_gap
North-Western  high_risk      0.8504           85.04             120.8             -77.4967          43.5135


# Save artefacts

In [ ]:

print("\n── Saving artefacts ─────────────────────────────────────")

# 1. Trained model
joblib.dump(best_model, "artefacts/xgb_v1.0.joblib")
print("Saved: artefacts/xgb_v1.0.joblib")

# 2. Province predictions
pred_df.to_csv("artefacts/province_predictions.csv", index=False)
print("Saved: artefacts/province_predictions.csv")

# 3. Full engineered dataset
df_model.to_csv("artefacts/province_features_engineered.csv", index=False)
print("Saved: artefacts/province_features_engineered.csv")

# 4. SHAP importance
shap_df.to_csv("artefacts/shap_importance.csv", index=False)
print("Saved: artefacts/shap_importance.csv")

# 5. Feature column list
with open("artefacts/feature_columns.json","w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print("Saved: artefacts/feature_columns.json")

# 6. Model metadata
lopo_auc_mean = round(lopo_df["auc"].mean(), 4) if len(lopo_df) else None
lopo_auc_std  = round(lopo_df["auc"].std(), 4)  if len(lopo_df) else None

model_meta = {
    "model_version":     "xgb_v1.0",
    "algorithm":         "XGBoost (RandomizedSearchCV, 60 iter, 5-fold stratified CV)",
    "level":             "provincial",
    "features":          FEATURE_COLS,
    "feature_labels":    FEATURE_LABELS,
    "risk_threshold":    0.65,
    "label_strategy":    "relative — top 30th percentile attrition proxy rate per year",
    "target_variable":   "attrition_proxy_rate (net teacher headcount decline / prior year count)",
    "f1_score":          round(test_f1, 4),
    "recall_score":      round(test_rec, 4),
    "precision_score":   round(test_prec, 4),
    "auc_score":         round(test_auc, 4) if not np.isnan(test_auc) else None,
    "lopo_auc_mean":     lopo_auc_mean,
    "lopo_auc_std":      lopo_auc_std,
    "cv_auc":            round(search.best_score_, 4),
    "train_years":       f"2009–{TRAIN_END}",
    "test_years":        f"{TRAIN_END+1}–{latest_year}",
    "missing_years":     "2015, 2017, 2018 (bulletins unavailable)",
    "provinces":         N_PROV,
    "artefact_path":     "artefacts/xgb_v1.0.joblib",
    "notes": (
        "Trained on MoE Education Statistics Bulletin data (2009-2025), "
        "primary level, provincial aggregate. "
        "attrition_count was inconsistently recorded across bulletin years "
        "and was therefore omitted; the target variable is derived from "
        "net teacher headcount decline (Ingersoll, 2001). "
        "School-level prediction is a planned future improvement "
        "contingent on access to EMIS school-level data from the "
        "Ministry of Education Permanent Secretary."
    ),
}
with open("artefacts/model_metadata.json","w") as f:
    json.dump(model_meta, f, indent=2)
print("Saved: artefacts/model_metadata.json")


── Saving artefacts ─────────────────────────────────────
Saved: artefacts/xgb_v1.0.joblib
Saved: artefacts/province_predictions.csv
Saved: artefacts/province_features_engineered.csv
Saved: artefacts/shap_importance.csv
Saved: artefacts/feature_columns.json
Saved: artefacts/model_metadata.json
